# PCE excess inflation contributions

Reconstructs aggregate PCE inflation from the 199 detailed components using a Törnqvist index: average expenditure shares at the start and end of the lookback window applied to log price relatives. This provides a close approximation to BEA's chained Fisher PCE price index and a natural component decomposition.

Each component is compared with its average inflation rate over the user-defined benchmark period. Signed component contributions are allocated from the additive log inflation gaps and rescaled so they sum exactly to reconstructed PCE inflation minus the reconstructed component benchmark.

In [ ]:
# User inputs
API_KEY = "FD4E7F46-5D89-4CA8-BE48-4F5997F83B3B"
LOOKBACK_MONTHS = 12  # for example: 12, 6, or 3
START_YEAR = 2000
BENCHMARK = "2000-2019"  # inclusive start and end years

CATEGORIES_FILE = "Detailed PCE.xlsx"

In [ ]:
import json
from urllib.parse import urlencode
from urllib.request import urlopen

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

if LOOKBACK_MONTHS <= 0:
    raise ValueError("LOOKBACK_MONTHS must be positive.")
try:
    benchmark_start_text, benchmark_end_text = (
        BENCHMARK.strip().strip(";").split("-", maxsplit=1)
    )
    BENCHMARK_START_YEAR = int(benchmark_start_text.strip())
    BENCHMARK_END_YEAR = int(benchmark_end_text.strip())
except (AttributeError, TypeError, ValueError) as error:
    raise ValueError("BENCHMARK must look like '2000-2019'.") from error
if BENCHMARK_END_YEAR < BENCHMARK_START_YEAR:
    raise ValueError("The benchmark end year must not precede the start year.")
benchmark_label = f"{BENCHMARK_START_YEAR}-{BENCHMARK_END_YEAR}"

required_columns = ["Line Item", "Description", "Code", "Major Group", "Classification"]
categories = pd.read_excel(CATEGORIES_FILE)
missing_columns = [column for column in required_columns if column not in categories.columns]
if missing_columns:
    raise ValueError(f"Missing required Excel columns: {missing_columns}")
categories = categories[required_columns].dropna(subset=["Line Item"]).copy()
categories["Line Item"] = pd.to_numeric(categories["Line Item"]).astype(int)
assert len(categories) == 199, f"Expected 199 categories, found {len(categories)}."
assert categories["Line Item"].is_unique, "Line Item values must be unique."
assert categories["Major Group"].notna().all(), "Every category must have a Major Group."
assert categories["Major Group"].nunique() == 10, "Expected 10 Major Groups."

def get_bea_table(dataset_name, table_name, years):
    params = {
        "UserID": API_KEY,
        "method": "GetData",
        "DataSetName": dataset_name,
        "TableName": table_name,
        "Frequency": "M",
        "Year": ",".join(map(str, years)),
        "ResultFormat": "JSON",
    }
    url = "https://apps.bea.gov/api/data/?" + urlencode(params)
    with urlopen(url, timeout=60) as response:
        payload = json.load(response)
    results = payload["BEAAPI"]["Results"]
    if "Data" not in results:
        raise RuntimeError(f"BEA API error: {results}")

    data = pd.DataFrame(results["Data"])
    data["Line Item"] = pd.to_numeric(data["LineNumber"], errors="coerce")
    data["Date"] = pd.PeriodIndex(
        data["TimePeriod"].str.replace("M", "-", regex=False), freq="M"
    )
    values = (
        data["DataValue"].astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("(", "-", regex=False)
        .str.replace(")", "", regex=False)
    )
    data["Value"] = pd.to_numeric(values, errors="coerce")
    data["Series Code"] = data.get("SeriesCode")
    return data[["Line Item", "Series Code", "Date", "Value"]]

current_year = pd.Timestamp.today().year
years_of_history = (LOOKBACK_MONTHS + 11) // 12
first_year = min(START_YEAR, BENCHMARK_START_YEAR) - years_of_history
years = range(first_year, current_year + 1)

prices = get_bea_table("NIUnderlyingDetail", "U20404", years)
spending = get_bea_table("NIUnderlyingDetail", "U20405", years)
official_prices = get_bea_table("NIPA", "T20804", years)

line_items = categories["Line Item"].tolist()
price_panel = (
    prices[prices["Line Item"].isin(line_items)]
    .pivot(index="Date", columns="Line Item", values="Value")
    .sort_index()
    .reindex(columns=line_items)
)
spending_panel = (
    spending[spending["Line Item"].isin(line_items)]
    .pivot(index="Date", columns="Line Item", values="Value")
    .sort_index()
    .reindex(columns=line_items)
)
official_price_index = (
    official_prices.loc[official_prices["Series Code"].eq("DPCERG")]
    .drop_duplicates(subset="Date")
    .set_index("Date")["Value"]
    .sort_index()
)
if official_price_index.empty:
    raise ValueError("Official PCE price index DPCERG was not found in BEA table T20804.")

component_price_relative = price_panel / price_panel.shift(LOOKBACK_MONTHS)
component_inflation_panel = (
    component_price_relative ** (12 / LOOKBACK_MONTHS) - 1
) * 100
official_pce_inflation_full = (
    (official_price_index / official_price_index.shift(LOOKBACK_MONTHS))
    ** (12 / LOOKBACK_MONTHS) - 1
) * 100

benchmark_dates = pd.period_range(
    f"{BENCHMARK_START_YEAR}-01", f"{BENCHMARK_END_YEAR}-12", freq="M"
)
expected_benchmark_observations = len(benchmark_dates)
benchmark_sample = component_inflation_panel.reindex(benchmark_dates)
benchmark_observations = benchmark_sample.notna().sum(axis=0)
benchmark_rates = benchmark_sample.mean(axis=0)
benchmark_column = f"{benchmark_label} average inflation"
if benchmark_rates.le(-100).any():
    raise ValueError("A component benchmark is at or below -100%, so it cannot be converted to a price relative.")
benchmark_price_relative = (
    1 + benchmark_rates / 100
) ** (LOOKBACK_MONTHS / 12)

component_benchmarks = categories.set_index("Line Item").copy()
component_benchmarks[benchmark_column] = benchmark_rates
component_benchmarks["Benchmark observations"] = benchmark_observations
incomplete_benchmarks = component_benchmarks.loc[
    component_benchmarks["Benchmark observations"].ne(expected_benchmark_observations)
]
if not incomplete_benchmarks.empty:
    display(incomplete_benchmarks)
    raise ValueError(
        f"{len(incomplete_benchmarks)} components do not have all "
        f"{expected_benchmark_observations} monthly benchmark observations."
    )

start_spending = spending_panel.shift(LOOKBACK_MONTHS)
valid = (
    component_price_relative.gt(0)
    & component_price_relative.notna()
    & spending_panel.notna()
    & start_spending.notna()
    & benchmark_price_relative.gt(0)
    & benchmark_price_relative.notna()
)
current_spending_total = spending_panel.where(valid).sum(axis=1, min_count=1)
start_spending_total = start_spending.where(valid).sum(axis=1, min_count=1)
current_shares = spending_panel.where(valid).div(current_spending_total, axis=0)
start_shares = start_spending.where(valid).div(start_spending_total, axis=0)
average_pce_weights = (current_shares + start_shares) / 2

current_log_relatives = np.log(component_price_relative.where(valid))
benchmark_log_relatives = np.log(benchmark_price_relative)
tornqvist_current_log_change = (
    average_pce_weights * current_log_relatives
).sum(axis=1, min_count=1)
tornqvist_benchmark_log_change = (
    average_pce_weights * benchmark_log_relatives
).sum(axis=1, min_count=1)
annualization_factor = 12 / LOOKBACK_MONTHS
reconstructed_pce_inflation_full = (
    np.exp(tornqvist_current_log_change * annualization_factor) - 1
) * 100
reconstructed_benchmark_full = (
    np.exp(tornqvist_benchmark_log_change * annualization_factor) - 1
) * 100
net_excess_contribution_full = (
    reconstructed_pce_inflation_full - reconstructed_benchmark_full
)

component_log_gaps = average_pce_weights * (
    current_log_relatives - benchmark_log_relatives
)
total_log_gap = component_log_gaps.sum(axis=1, min_count=1)
contribution_scale = net_excess_contribution_full.div(total_log_gap)
component_contributions_full = component_log_gaps.mul(contribution_scale, axis=0)
near_zero_gap = total_log_gap.abs().lt(1e-14)
component_contributions_full.loc[near_zero_gap] = 0.0

start_period = pd.Period(f"{START_YEAR}-01", freq="M")
dates = price_panel.index.intersection(spending_panel.index)
dates = dates[dates >= start_period]
component_inflation = component_inflation_panel.reindex(index=dates, columns=line_items)
inflation_gap = component_inflation.sub(benchmark_rates, axis="columns")
average_pce_weights = average_pce_weights.reindex(index=dates, columns=line_items)
component_contributions = component_contributions_full.reindex(index=dates, columns=line_items)
component_contributions.index.name = "Date"

reconstructed_pce_inflation = reconstructed_pce_inflation_full.reindex(dates)
official_pce_inflation = official_pce_inflation_full.reindex(dates)
reconstructed_benchmark = reconstructed_benchmark_full.reindex(dates)
net_excess_contribution = net_excess_contribution_full.reindex(dates)
reconstruction_difference = reconstructed_pce_inflation - official_pce_inflation

excess_inflation_time_series = pd.DataFrame({
    "Reconstructed PCE inflation": reconstructed_pce_inflation,
    "Official BEA PCE inflation": official_pce_inflation,
    "Reconstruction difference": reconstruction_difference,
    "Reconstructed component benchmark": reconstructed_benchmark,
    "Net excess inflation contribution": net_excess_contribution,
})
excess_inflation_time_series.index.name = "Date"

print(
    f"{len(excess_inflation_time_series)} monthly observations: "
    f"{excess_inflation_time_series.index.min()} to {excess_inflation_time_series.index.max()}"
)
print(
    f"Component benchmarks use {expected_benchmark_observations} monthly observations "
    f"from {benchmark_dates.min()} through {benchmark_dates.max()}."
)

## 1. Component benchmarks

Each component benchmark is the arithmetic average of its annualized inflation rates over the inclusive period selected in `BENCHMARK`.

In [ ]:
benchmark_summary = pd.DataFrame({
    "Value": [benchmark_label, f"{LOOKBACK_MONTHS} months", expected_benchmark_observations]
}, index=[
    "Benchmark period",
    "Inflation horizon",
    "Monthly observations per component",
])
display(benchmark_summary)

component_benchmarks_display = component_benchmarks.copy()
component_benchmarks_display[benchmark_column] = (
    component_benchmarks_display[benchmark_column].map("{:.3f}%".format)
)
display(component_benchmarks_display)

## 2. Reconstructed and official PCE inflation

The reconstructed rate uses the 199 detailed components and Törnqvist aggregation. The official BEA PCE series is shown as a control; their small difference reflects exact Fisher chaining, aggregation detail, and other BEA adjustments.

In [ ]:
latest_excess_display = excess_inflation_time_series.tail(12).copy()
for column in latest_excess_display.columns:
    latest_excess_display[column] = latest_excess_display[column].map("{:+.3f}%".format)
display(latest_excess_display)

try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Install matplotlib to display the charts.")
else:
    ax = excess_inflation_time_series[[
        "Reconstructed PCE inflation", "Official BEA PCE inflation"
    ]].plot(figsize=(12, 4), linewidth=1.3)
    ax.set_title(f"Reconstructed and official PCE inflation ({LOOKBACK_MONTHS}-month measure)")
    ax.set_xlabel("Date")
    ax.set_ylabel("Percent")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

    ax = excess_inflation_time_series["Net excess inflation contribution"].plot(
        figsize=(12, 4), linewidth=1.5, color="#9c2f2f"
    )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"Net excess inflation relative to {benchmark_label} component benchmarks")
    ax.set_xlabel("Date")
    ax.set_ylabel("Percentage points")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

## 3. Latest component contributions

`latest_component_contributions` contains all components and is sorted by absolute contribution. Average endpoint weights are the mean of each component's expenditure share at the start and end of the lookback window.

In [ ]:
latest_date = excess_inflation_time_series.index.max()
latest_component_contributions = categories.set_index("Line Item").copy()
latest_component_contributions["Current inflation rate"] = component_inflation.loc[latest_date]
latest_component_contributions["Component benchmark"] = benchmark_rates
latest_component_contributions["Inflation gap"] = inflation_gap.loc[latest_date]
latest_component_contributions["Average endpoint PCE weight"] = average_pce_weights.loc[latest_date]
latest_component_contributions["Excess inflation contribution"] = (
    component_contributions.loc[latest_date]
)
latest_component_contributions["Absolute contribution"] = (
    latest_component_contributions["Excess inflation contribution"].abs()
)
latest_component_contributions = (
    latest_component_contributions.sort_values("Absolute contribution", ascending=False)
    .drop(columns="Absolute contribution")
)

print(f"Largest component contributions in {latest_date}:")
latest_components_display = latest_component_contributions.head(30).copy()
for column in ["Current inflation rate", "Component benchmark"]:
    latest_components_display[column] = latest_components_display[column].map("{:.3f}%".format)
latest_components_display["Inflation gap"] = (
    latest_components_display["Inflation gap"].map("{:+.3f} pp".format)
)
latest_components_display["Average endpoint PCE weight"] = (
    latest_components_display["Average endpoint PCE weight"].map("{:.3%}".format)
)
latest_components_display["Excess inflation contribution"] = (
    latest_components_display["Excess inflation contribution"].map("{:+.4f} pp".format)
)
display(latest_components_display)

## 4. Contributions by Major Group

Major Group contributions sum the signed component contributions and reconcile exactly to reconstructed PCE inflation minus the reconstructed component benchmark.

In [ ]:
group_order = categories["Major Group"].drop_duplicates().tolist()
line_to_group = categories.set_index("Line Item")["Major Group"]
column_groups = line_to_group.reindex(line_items)
major_group_contributions = pd.DataFrame(index=dates)
for group in group_order:
    group_columns = column_groups.index[column_groups.eq(group)]
    major_group_contributions[group] = (
        component_contributions[group_columns].sum(axis=1, min_count=1)
    )
major_group_contributions.index.name = "Date"

latest_major_group_contributions = pd.DataFrame({
    "Excess inflation contribution": major_group_contributions.loc[latest_date]
})
latest_major_group_contributions.loc["Total"] = (
    latest_major_group_contributions.sum(axis=0)
)
print(f"Major Group contributions for {latest_date}:")
latest_major_group_display = latest_major_group_contributions.copy()
latest_major_group_display["Excess inflation contribution"] = (
    latest_major_group_display["Excess inflation contribution"].map("{:+.4f} pp".format)
)
display(latest_major_group_display)

try:
    import matplotlib.pyplot as plt
except ImportError:
    pass
else:
    latest_group_plot = latest_major_group_contributions.drop(index="Total").sort_values(
        "Excess inflation contribution"
    )
    colors = np.where(
        latest_group_plot["Excess inflation contribution"].ge(0),
        "#9c2f2f", "#2f5d8c"
    )
    ax = latest_group_plot["Excess inflation contribution"].plot.barh(
        figsize=(9, 5), color=colors
    )
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"Major Group excess inflation contributions, {latest_date}")
    ax.set_xlabel("Percentage points")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

## 5. Core PCE analysis

The Core classification in `Detailed PCE.xlsx` excludes all rows marked Food & energy. The same Törnqvist reconstruction and benchmark methodology are then applied to the remaining components, with BEA's official core PCE price index shown as a control.

In [ ]:
core_categories = categories.loc[categories["Classification"].eq("Core")].copy()
core_line_items = core_categories["Line Item"].tolist()
assert len(core_line_items) == 171, f"Expected 171 Core components, found {len(core_line_items)}."

core_valid = valid[core_line_items]
core_start_spending = start_spending[core_line_items]
core_current_spending = spending_panel[core_line_items]
core_current_total = core_current_spending.where(core_valid).sum(axis=1, min_count=1)
core_start_total = core_start_spending.where(core_valid).sum(axis=1, min_count=1)
core_current_shares = core_current_spending.where(core_valid).div(core_current_total, axis=0)
core_start_shares = core_start_spending.where(core_valid).div(core_start_total, axis=0)
core_average_pce_weights_full = (core_current_shares + core_start_shares) / 2

core_current_log_relatives = current_log_relatives[core_line_items]
core_benchmark_log_relatives = benchmark_log_relatives.reindex(core_line_items)
core_current_log_change = (
    core_average_pce_weights_full * core_current_log_relatives
).sum(axis=1, min_count=1)
core_benchmark_log_change = (
    core_average_pce_weights_full * core_benchmark_log_relatives
).sum(axis=1, min_count=1)
core_reconstructed_inflation_full = (
    np.exp(core_current_log_change * annualization_factor) - 1
) * 100
core_reconstructed_benchmark_full = (
    np.exp(core_benchmark_log_change * annualization_factor) - 1
) * 100
core_net_excess_full = (
    core_reconstructed_inflation_full - core_reconstructed_benchmark_full
)

core_component_log_gaps = core_average_pce_weights_full * (
    core_current_log_relatives - core_benchmark_log_relatives
)
core_total_log_gap = core_component_log_gaps.sum(axis=1, min_count=1)
core_contribution_scale = core_net_excess_full.div(core_total_log_gap)
core_component_contributions_full = core_component_log_gaps.mul(
    core_contribution_scale, axis=0
)
core_near_zero_gap = core_total_log_gap.abs().lt(1e-14)
core_component_contributions_full.loc[core_near_zero_gap] = 0.0

official_core_price_index = (
    official_prices.loc[official_prices["Series Code"].eq("DPCCRG")]
    .drop_duplicates(subset="Date")
    .set_index("Date")["Value"]
    .sort_index()
)
if official_core_price_index.empty:
    raise ValueError("Official core PCE price index DPCCRG was not found in BEA table T20804.")
official_core_inflation_full = (
    (official_core_price_index / official_core_price_index.shift(LOOKBACK_MONTHS))
    ** annualization_factor - 1
) * 100

core_average_pce_weights = core_average_pce_weights_full.reindex(
    index=dates, columns=core_line_items
)
core_component_contributions = core_component_contributions_full.reindex(
    index=dates, columns=core_line_items
)
core_component_contributions.index.name = "Date"
core_reconstructed_inflation = core_reconstructed_inflation_full.reindex(dates)
core_reconstructed_benchmark = core_reconstructed_benchmark_full.reindex(dates)
core_net_excess = core_net_excess_full.reindex(dates)
official_core_inflation = official_core_inflation_full.reindex(dates)
core_reconstruction_difference = core_reconstructed_inflation - official_core_inflation

core_pce_time_series = pd.DataFrame({
    "Reconstructed core PCE inflation": core_reconstructed_inflation,
    "Official BEA core PCE inflation": official_core_inflation,
    "Core reconstruction difference": core_reconstruction_difference,
    "Reconstructed core component benchmark": core_reconstructed_benchmark,
    "Net core excess inflation contribution": core_net_excess,
})
core_pce_time_series.index.name = "Date"

latest_core_display = core_pce_time_series.tail(12).copy()
for column in latest_core_display.columns:
    if "difference" in column.lower() or "contribution" in column.lower():
        latest_core_display[column] = latest_core_display[column].map("{:+.3f} pp".format)
    else:
        latest_core_display[column] = latest_core_display[column].map("{:.3f}%".format)
display(latest_core_display)

try:
    import matplotlib.pyplot as plt
except ImportError:
    pass
else:
    ax = core_pce_time_series[[
        "Reconstructed core PCE inflation",
        "Official BEA core PCE inflation",
    ]].plot(figsize=(12, 4), linewidth=1.3)
    ax.set_title(f"Reconstructed and official core PCE inflation ({LOOKBACK_MONTHS}-month measure)")
    ax.set_xlabel("Date")
    ax.set_ylabel("Percent")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

    ax = core_pce_time_series["Net core excess inflation contribution"].plot(
        figsize=(12, 4), linewidth=1.5, color="#6b3f8c"
    )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"Net core excess inflation relative to {benchmark_label} component benchmarks")
    ax.set_xlabel("Date")
    ax.set_ylabel("Percentage points")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

### 5.1 Latest core component and Major Group contributions

In [ ]:
latest_core_component_contributions = core_categories.set_index("Line Item").copy()
latest_core_component_contributions["Current inflation rate"] = (
    component_inflation.loc[latest_date, core_line_items]
)
latest_core_component_contributions["Component benchmark"] = benchmark_rates.reindex(core_line_items)
latest_core_component_contributions["Inflation gap"] = (
    inflation_gap.loc[latest_date, core_line_items]
)
latest_core_component_contributions["Average endpoint core PCE weight"] = (
    core_average_pce_weights.loc[latest_date]
)
latest_core_component_contributions["Core excess inflation contribution"] = (
    core_component_contributions.loc[latest_date]
)
latest_core_component_contributions["Absolute contribution"] = (
    latest_core_component_contributions["Core excess inflation contribution"].abs()
)
latest_core_component_contributions = (
    latest_core_component_contributions.sort_values("Absolute contribution", ascending=False)
    .drop(columns="Absolute contribution")
)

print(f"Largest core component contributions in {latest_date}:")
latest_core_components_display = latest_core_component_contributions.head(30).copy()
for column in ["Current inflation rate", "Component benchmark"]:
    latest_core_components_display[column] = (
        latest_core_components_display[column].map("{:.3f}%".format)
    )
latest_core_components_display["Inflation gap"] = (
    latest_core_components_display["Inflation gap"].map("{:+.3f} pp".format)
)
latest_core_components_display["Average endpoint core PCE weight"] = (
    latest_core_components_display["Average endpoint core PCE weight"].map("{:.3%}".format)
)
latest_core_components_display["Core excess inflation contribution"] = (
    latest_core_components_display["Core excess inflation contribution"].map("{:+.4f} pp".format)
)
display(latest_core_components_display)

core_column_groups = line_to_group.reindex(core_line_items)
core_major_group_contributions = pd.DataFrame(index=dates)
for group in group_order:
    group_columns = core_column_groups.index[core_column_groups.eq(group)]
    core_major_group_contributions[group] = (
        core_component_contributions[group_columns].sum(axis=1, min_count=1)
    )
core_major_group_contributions.index.name = "Date"
latest_core_major_group_contributions = pd.DataFrame({
    "Core excess inflation contribution": core_major_group_contributions.loc[latest_date]
})
latest_core_major_group_contributions.loc["Total"] = (
    latest_core_major_group_contributions.sum(axis=0)
)
print(f"Core Major Group contributions for {latest_date}:")
latest_core_group_display = latest_core_major_group_contributions.copy()
latest_core_group_display["Core excess inflation contribution"] = (
    latest_core_group_display["Core excess inflation contribution"].map("{:+.4f} pp".format)
)
display(latest_core_group_display)

try:
    import matplotlib.pyplot as plt
except ImportError:
    pass
else:
    core_group_plot = latest_core_major_group_contributions.drop(index="Total").sort_values(
        "Core excess inflation contribution"
    )
    colors = np.where(
        core_group_plot["Core excess inflation contribution"].ge(0),
        "#6b3f8c", "#2f5d8c"
    )
    ax = core_group_plot["Core excess inflation contribution"].plot.barh(
        figsize=(9, 5), color=colors
    )
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"Core Major Group excess inflation contributions, {latest_date}")
    ax.set_xlabel("Percentage points")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

## 6. Validation and spot checks

In [ ]:
weight_sum_error = (average_pce_weights.sum(axis=1, min_count=1) - 1).abs().max()
component_reconciliation_error = (
    component_contributions.sum(axis=1, min_count=1) - net_excess_contribution
).abs().max()
group_reconciliation_error = (
    major_group_contributions.sum(axis=1, min_count=1) - net_excess_contribution
).abs().max()
identity_error = (
    reconstructed_pce_inflation
    - reconstructed_benchmark
    - net_excess_contribution
).abs().max()
latest_reconstruction_difference = reconstruction_difference.loc[latest_date]
core_weight_sum_error = (
    core_average_pce_weights.sum(axis=1, min_count=1) - 1
).abs().max()
core_component_reconciliation_error = (
    core_component_contributions.sum(axis=1, min_count=1) - core_net_excess
).abs().max()
core_group_reconciliation_error = (
    core_major_group_contributions.sum(axis=1, min_count=1) - core_net_excess
).abs().max()
core_identity_error = (
    core_reconstructed_inflation
    - core_reconstructed_benchmark
    - core_net_excess
).abs().max()
latest_core_reconstruction_difference = core_reconstruction_difference.loc[latest_date]

assert weight_sum_error < 1e-10
assert component_reconciliation_error < 1e-10
assert group_reconciliation_error < 1e-10
assert identity_error < 1e-10
assert abs(latest_reconstruction_difference) < 0.10
assert core_weight_sum_error < 1e-10
assert core_component_reconciliation_error < 1e-10
assert core_group_reconciliation_error < 1e-10
assert core_identity_error < 1e-10
assert abs(latest_core_reconstruction_difference) < 0.10

spot_line_item = latest_component_contributions.index[0]
spot_component_expected = component_contributions_full.loc[latest_date, spot_line_item]
spot_component_actual = component_contributions.loc[latest_date, spot_line_item]
assert np.isclose(spot_component_actual, spot_component_expected)

spot_group = group_order[0]
spot_group_columns = column_groups.index[column_groups.eq(spot_group)]
spot_group_expected = component_contributions.loc[latest_date, spot_group_columns].sum()
spot_group_actual = major_group_contributions.loc[latest_date, spot_group]
assert np.isclose(spot_group_actual, spot_group_expected)

validation_summary = pd.DataFrame({
    "Check": [
        "Mapped categories",
        "Major Groups",
        "Benchmark observations per component",
        "Maximum average-weight sum error",
        "Maximum component reconciliation error",
        "Maximum Major Group reconciliation error",
        "Maximum contribution identity error",
        "Latest reconstruction difference",
        "Core components",
        "Maximum core average-weight sum error",
        "Maximum core component reconciliation error",
        "Maximum core Major Group reconciliation error",
        "Maximum core contribution identity error",
        "Latest core reconstruction difference",
        f"Spot-checked component {spot_line_item}",
        f"Spot-checked Major Group: {spot_group}",
    ],
    "Result": [
        len(categories),
        categories["Major Group"].nunique(),
        expected_benchmark_observations,
        weight_sum_error,
        component_reconciliation_error,
        group_reconciliation_error,
        identity_error,
        f"{latest_reconstruction_difference:+.4f} pp",
        len(core_categories),
        core_weight_sum_error,
        core_component_reconciliation_error,
        core_group_reconciliation_error,
        core_identity_error,
        f"{latest_core_reconstruction_difference:+.4f} pp",
        "Pass",
        "Pass",
    ],
})
display(validation_summary)